# 01. Скрейпинг статей
**Вход:** CSV из Медиалогии (`;`-separated, UTF-8-sig)  
**Выход:** `scraped_articles.csv` — те же поля + `text`, `scrape_status`, `http_code`  
**Источники:** РБК, Коммерсантъ, Ведомости  

Запуск:
1. Ячейка 1 — установка пакетов
2. Ячейка 2 — импорт, настройки и функции
3. Ячейка 3 — диагностика чекпоинта
4. Ячейка 4 — основной прогон

In [ ]:
# ── ЯЧЕЙКА 1: Установка пакетов ──────────────────────────────
!pip install requests beautifulsoup4 lxml tqdm -q

In [ ]:
# ── ЯЧЕЙКА 2: Настройки и функции ────────────────────────────
import os, time, random, re
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from tqdm.notebook import tqdm

# ── Пути ─────────────────────────────────────────────────────
INPUT_FILE      = '/content/medialogy_export.csv'   # исходный файл из Медиалогии
CHECKPOINT_FILE = '/content/scraped_checkpoint.csv'
OUTPUT_FILE     = '/content/scraped_articles.csv'

# ── Параметры ─────────────────────────────────────────────────
DELAY_MIN  = 2.0    # пауза между запросами, сек
DELAY_MAX  = 4.0
BATCH_SAVE = 200    # сохранять чекпоинт каждые N записей
TIMEOUT    = 15

# ── Целевые домены ────────────────────────────────────────────
TARGET_DOMAINS = {
    'www.rbc.ru', 'pro.rbc.ru', 'tv.rbc.ru',
    'kuban.rbc.ru', 'rostov.rbc.ru', 'ufa.rbc.ru',
    'realty.rbc.ru', 't.rbc.ru', 'companies.rbc.ru',
    'www.kommersant.ru',
    'www.vedomosti.ru', 'vedomosti.ru', 'spb.vedomosti.ru',
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Connection': 'keep-alive',
}

# ── Константа для обрезки рекламного глоссария РБК ───────────
RBC_TRIM_RE = re.compile(
    r'(покупка ценных бумаг|вложение денежных средств|различают финансовые|'
    r'делятся множество подвидов)',
    re.IGNORECASE
)

# ═══════════════════════════════════════════════════════════════
# Парсеры
# ═══════════════════════════════════════════════════════════════

def parse_rbc(soup):
    """РБК: www.rbc.ru и поддомены."""
    # Пейволл
    if soup.find('div', class_='article__content-lock'):
        return None, 'paywall'
    block = (soup.find('div', class_='article__text') or
             soup.find('div', class_='article__text__overview') or
             soup.find('div', {'itemprop': 'articleBody'}))
    if not block:
        return None, 'parse_failed'
    text = block.get_text(' ', strip=True)
    # Обрезаем глоссарий
    m = RBC_TRIM_RE.search(text)
    if m:
        text = text[:m.start()].strip()
    return text, 'ok'

def parse_rbc_companies(soup):
    """companies.rbc.ru — отдельная вёрстка."""
    if soup.find('div', class_='news-detail-paywall__container'):
        return None, 'paywall'
    block = soup.find('div', class_='news-detail__content')
    if not block:
        return None, 'parse_failed'
    return block.get_text(' ', strip=True), 'ok'

def parse_kommersant(soup):
    """Коммерсантъ."""
    if soup.find('div', class_='paywall') or soup.find('div', class_='subscription-block'):
        return None, 'paywall'
    block = (soup.find('div', class_='article_text_wrapper') or
             soup.find('div', itemprop='articleBody') or
             soup.find('div', class_='b-article__text'))
    if not block:
        return None, 'parse_failed'
    return block.get_text(' ', strip=True), 'ok'

def parse_vedomosti(soup):
    """Ведомости."""
    if (soup.find('div', class_='paywall') or
            soup.find('div', class_='article-premium') or
            soup.find('div', class_='article__paywall')):
        return None, 'paywall'
    block = (soup.find('div', class_='article-body') or
             soup.find('div', itemprop='articleBody') or
             soup.find('article', class_='article'))
    if not block:
        return None, 'parse_failed'
    return block.get_text(' ', strip=True), 'ok'

DOMAIN_PARSERS = {
    'www.rbc.ru':        parse_rbc,
    'pro.rbc.ru':        parse_rbc,
    'tv.rbc.ru':         parse_rbc,
    'kuban.rbc.ru':      parse_rbc,
    'rostov.rbc.ru':     parse_rbc,
    'ufa.rbc.ru':        parse_rbc,
    'realty.rbc.ru':     parse_rbc,
    't.rbc.ru':          parse_rbc,
    'companies.rbc.ru':  parse_rbc_companies,
    'www.kommersant.ru': parse_kommersant,
    'www.vedomosti.ru':  parse_vedomosti,
    'vedomosti.ru':      parse_vedomosti,
    'spb.vedomosti.ru':  parse_vedomosti,
}

# ═══════════════════════════════════════════════════════════════
# Основная функция скрейпинга
# ═══════════════════════════════════════════════════════════════

def scrape_url(url):
    domain = urlparse(url).netloc
    parser_fn = DOMAIN_PARSERS.get(domain)
    if parser_fn is None:
        return {'text': None, 'status': 'skip', 'http_code': None}
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if r.status_code != 200:
            return {'text': None, 'status': 'error', 'http_code': r.status_code}
        soup = BeautifulSoup(r.text, 'lxml')
        text, status = parser_fn(soup)
        return {'text': text, 'status': status, 'http_code': r.status_code}
    except requests.exceptions.Timeout:
        return {'text': None, 'status': 'timeout', 'http_code': None}
    except Exception as e:
        return {'text': None, 'status': 'error', 'http_code': str(e)[:50]}

print('✓ Функции загружены')

In [ ]:
# ── ЯЧЕЙКА 3: Диагностика входных данных и чекпоинта ─────────
# Запускай перед основным прогоном — убедись что всё нашлось.

df_raw = pd.read_csv(
    INPUT_FILE,
    sep=';',
    engine='c',
    lineterminator='\n',
    encoding='utf-8-sig',
    on_bad_lines='skip',
)
# Нормализуем имена колонок
df_raw.columns = [c.strip().lower() for c in df_raw.columns]
df_raw['url_clean'] = df_raw['url'].astype(str).str.strip()
df_raw['domain']    = df_raw['url_clean'].apply(lambda x: urlparse(x).netloc)
df_target = df_raw[df_raw['domain'].isin(TARGET_DOMAINS)].copy()

print(f'Всего строк в файле:   {len(df_raw):>7}')
print(f'Целевых доменов:       {len(df_target):>7}')
print()
print(df_target['domain'].value_counts().to_string())
print()

# Чекпоинт
if os.path.exists(CHECKPOINT_FILE):
    ckpt = pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig')
    done = set(ckpt['url_clean'])
    print(f'✓ Чекпоинт найден: {len(ckpt)} записей')
    print(f'  Статусы: {ckpt["scrape_status"].value_counts().to_dict()}')
    print(f'  Осталось: {len(df_target) - len(done)}')
else:
    print('✗ Чекпоинт не обнаржен, запуск с начала списка')

In [ ]:
# ── ЯЧЕЙКА 4: Основной прогон ─────────────────────────────────

# Определяем что уже сделано
already_done = set()
if os.path.exists(CHECKPOINT_FILE):
    already_done = set(pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig')['url_clean'])

to_scrape = df_target[~df_target['url_clean'].isin(already_done)].reset_index(drop=True)
print(f'Осталось скрейпить: {len(to_scrape)}')

results = []
for i, row in enumerate(tqdm(to_scrape.itertuples(), total=len(to_scrape))):
    res = scrape_url(row.url_clean)
    results.append({
        'num':           row.num,
        'title':         row.title,
        'date':          row.date,
        'media':         row.media,
        'media_index':   row.media_index,
        'visibility':    row.visibility,
        'url_clean':     row.url_clean,
        'domain':        row.domain,
        'text':          res['text'],
        'scrape_status': res['status'],
        'http_code':     res['http_code'],
    })
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    # Сохранение чекпоинта каждые BATCH_SAVE записей
    if (i + 1) % BATCH_SAVE == 0:
        batch = pd.DataFrame(results)
        if os.path.exists(CHECKPOINT_FILE):
            batch = pd.concat([pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig'), batch],
                              ignore_index=True)
        batch.to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8-sig')
        results = []  # очищаем — всё в файле
        ok  = (batch['scrape_status'] == 'ok').sum()
        pw  = (batch['scrape_status'] == 'paywall').sum()
        err = batch['scrape_status'].isin(['error', 'parse_failed', 'timeout']).sum()
        print(f'[{i+1}/{len(to_scrape)}]  ok:{ok}  paywall:{pw}  err:{err}')

# Финальное сохранение (остаток)
if results:
    final = pd.DataFrame(results)
    if os.path.exists(CHECKPOINT_FILE):
        final = pd.concat([pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig'), final],
                          ignore_index=True)
    final = final.drop_duplicates(subset='url_clean')
    final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f'\n✓ Готово. Итого записей: {len(final)}')
    print(final['scrape_status'].value_counts().to_string())
elif os.path.exists(CHECKPOINT_FILE):
    final = pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig').drop_duplicates(subset='url_clean')
    final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f'\n✓ Готово. Итого записей: {len(final)}')
    print(final['scrape_status'].value_counts().to_string())